# Color-Invariant Saree Design Recognition
### AI Engineering Technical Challenge — DeepLure Saree Corpus & Indian Saree Patterns

This notebook provides an end-to-end PyTorch implementation covering:
1. **Textile Palette & Colorway Synthesis Engine**
2. **CIS-Net (Color-Invariant Saree Network)** with fixed Sobel/Laplacian gradient stem and IBN-ResNet backbone
3. **GeM Pooling & Hyperspherical Metric Learning Head** with ArcFace, SupCon, and Palette Decorrelation loss
4. **Complete Evaluation Protocol**: Identification (Top-1, Top-5, mAP, CMC) and Verification (ROC-AUC, EER, Color Bias Gap)
5. **Efficiency Benchmarking**: Parameter count, FLOPs, Latency, and Memory Footprint.

In [ ]:
# 1. Environment Setup & Dependencies
!pip install -q torch torchvision scikit-learn matplotlib tqdm
import os, math, time, json, random
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T
from sklearn.metrics import roc_curve, auc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

## 2. Approach Note (<= 500 Characters)
> **"CIS-Net uses an IBN-ResNet backbone with a fixed multi-gradient stem (Sobel & Laplacian) and GeM pooling to decouple textile motif geometry from chromatic palettes. Preprocessing extracts structural gradients and instance-normalizes RGB channels. Training uses balanced P×K sampling (8 designs × 4 colorways) with ArcFace loss (m=0.35, s=30), SupCon, and palette decorrelation under aggressive hue/channel jitter. Post-processing computes L2-normalized 512-d embeddings for cosine retrieval and verification."**

In [ ]:
# 3. Textile Colorway Engine & Authentic Palettes
TEXTILE_PALETTES = [
    [(180, 20, 40), (220, 180, 60), (240, 210, 120)],   # Crimson & Antique Gold
    [(20, 110, 60), (190, 30, 50), (215, 175, 70)],     # Emerald Green & Ruby
    [(15, 30, 95), (200, 210, 220), (240, 245, 255)],   # Royal Midnight Blue & Silver Zari
    [(225, 160, 30), (25, 45, 110), (240, 230, 200)],   # Mustard Yellow & Indigo Navy
    [(210, 30, 120), (10, 140, 140), (235, 200, 80)],   # Rani Pink & Peacock Teal
    [(45, 50, 55), (140, 145, 150), (220, 225, 230)],   # Monochrome Slate & Charcoal
]

def apply_palette_transfer(image: Image.Image, palette) -> Image.Image:
    gray = np.array(image.convert('L'), dtype=np.float32) / 255.0
    h, w = gray.shape
    p_low, p_high = np.percentile(gray, 2), np.percentile(gray, 98)
    norm_gray = np.clip((gray - p_low) / (p_high - p_low + 1e-6), 0.0, 1.0)
    c0, c1, c2 = [np.array(c, dtype=np.float32)/255.0 for c in palette]
    out_rgb = np.zeros((h, w, 3), dtype=np.float32)
    mask1 = norm_gray <= 0.5
    t1 = norm_gray[mask1] / 0.5
    for ch in range(3):
        out_rgb[mask1, ch] = (1.0 - t1) * c0[ch] + t1 * c1[ch]
    mask2 = norm_gray > 0.5
    t2 = (norm_gray[mask2] - 0.5) / 0.5
    for ch in range(3):
        out_rgb[mask2, ch] = (1.0 - t2) * c1[ch] + t2 * c2[ch]
    return Image.fromarray((out_rgb * 255.0).astype(np.uint8))
print('Colorway synthesis engine ready.')

In [ ]:
# 4. CIS-Net Model Architecture
class GradientStructureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        sx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).unsqueeze(0).unsqueeze(0)
        sy = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).unsqueeze(0).unsqueeze(0)
        lap = torch.tensor([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]).unsqueeze(0).unsqueeze(0)
        self.register_buffer('sobel_x', sx)
        self.register_buffer('sobel_y', sy)
        self.register_buffer('laplacian', lap)
    def forward(self, gray):
        gx = F.conv2d(gray, self.sobel_x, padding=1)
        gy = F.conv2d(gray, self.sobel_y, padding=1)
        mag = torch.sqrt(gx**2 + gy**2 + 1e-6)
        lap = F.conv2d(gray, self.laplacian, padding=1)
        return torch.cat([mag, lap, gray], dim=1)

class IBNConv(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.split_c = out_c // 2
        self.in_norm = nn.InstanceNorm2d(self.split_c, affine=True)
        self.bn_norm = nn.BatchNorm2d(out_c - self.split_c)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        x = self.conv(x)
        return self.act(torch.cat([self.in_norm(x[:, :self.split_c]), self.bn_norm(x[:, self.split_c:])], dim=1))

class GeMPooling(nn.Module):
    def __init__(self, p=3.0):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
    def forward(self, x):
        p_c = self.p.clamp(min=1.0, max=10.0)
        return F.avg_pool2d(x.clamp(min=1e-6).pow(p_c), (x.size(-2), x.size(-1))).pow(1.0 / p_c).squeeze(-1).squeeze(-1)

class CISNet(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.grad_ext = GradientStructureExtractor()
        self.stem = nn.Sequential(nn.Conv2d(6, 48, 5, stride=2, padding=2, bias=False), nn.BatchNorm2d(48), nn.SiLU(True), nn.MaxPool2d(3, 2, 1))
        self.stage1 = nn.Sequential(IBNConv(48, 64), IBNConv(64, 64))
        self.stage2 = nn.Sequential(IBNConv(64, 128, stride=2), IBNConv(128, 128))
        self.stage3 = nn.Sequential(nn.Conv2d(128, 256, 3, 2, 1, bias=False), nn.BatchNorm2d(256), nn.SiLU(True), nn.Conv2d(256, 256, 3, 1, 1, bias=False), nn.BatchNorm2d(256), nn.SiLU(True))
        self.stage4 = nn.Sequential(nn.Conv2d(256, 384, 3, 2, 1, bias=False), nn.BatchNorm2d(384), nn.SiLU(True))
        self.gem = GeMPooling(p=3.0)
        self.head = nn.Sequential(nn.Linear(384, embedding_dim, bias=False), nn.BatchNorm1d(embedding_dim), nn.SiLU(True), nn.Linear(embedding_dim, embedding_dim, bias=False), nn.BatchNorm1d(embedding_dim))
    def forward(self, x):
        gray = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]
        in_rgb = F.instance_norm(x, eps=1e-5)
        feat = self.stem(torch.cat([self.grad_ext(gray), in_rgb], dim=1))
        feat = self.stage4(self.stage3(self.stage2(self.stage1(feat))))
        return F.normalize(self.head(self.gem(feat)), p=2, dim=1)
print('CIS-Net architecture ready. Parameters:', sum(p.numel() for p in CISNet().parameters()))

In [ ]:
# 5. ArcFace Metric Learning Loss
class ArcFaceHead(nn.Module):
    def __init__(self, embedding_dim, num_classes, scale=30.0, margin=0.35):
        super().__init__()
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin
    def forward(self, emb, labels):
        cosine = F.linear(emb, F.normalize(self.weight, p=2, dim=1)).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(min=1e-7)
        phi = torch.where(cosine > self.th, cosine * self.cos_m - sine * self.sin_m, cosine - self.mm)
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.view(-1, 1).long(), 1.0)
        return (one_hot * phi + (1.0 - one_hot) * cosine) * self.scale
print('ArcFace head ready.')

In [ ]:
# 6. Efficiency & Latency Benchmark
model = CISNet().to(device).eval()
x_dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    for _ in range(10): model(x_dummy)
    t0 = time.perf_counter()
    for _ in range(50): model(x_dummy)
    t1 = time.perf_counter()
lat_ms = (t1 - t0) / 50 * 1000.0
params = sum(p.numel() for p in model.parameters())
print(f'Model Parameters: {params:,} ({params/1e6:.2f}M)')
print(f'Inference Latency: {lat_ms:.2f} ms/image ({1000.0/lat_ms:.1f} FPS)')
print(f'Embedding Footprint: 512-dim float32 (2 KB per saree)')